In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("InferSchema"). \
getOrCreate()

In [ ]:
df = spark.read \
.format("csv") \
.option("header", "false") \
.option("inferSchema", "true") \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")
#0.6s

In [ ]:
df2 = spark.read \
.format("csv") \
.option("header", "false") \
.option("inferSchema", "true") \
.option("samplingRatio", .1) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")
#0.3s

In [18]:
df3 = spark.read \
.format("csv") \
.option("header", "false") \
.option("inferSchema", "true") \
.option("samplingRatio", .0000000001) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")
#0.2s

In [ ]:
#Khong dung infer
df1 = spark.read \
.format("csv") \
.option("header", "false") \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")
#0.1s

In [9]:
df1.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



In [10]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: timestamp (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: string (nullable = true)



In [16]:
df2.printSchema()
#Khá chính xác

root
 |-- _c0: integer (nullable = true)
 |-- _c1: timestamp (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: string (nullable = true)



In [20]:
df3.printSchema()
#Không chính xác vif samplingRatio quá nhỏ, nên spark scan file quá ít, không hiểu đc context

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



In [11]:
df.show(5)

+---+-------------------+-----+---------------+
|_c0|                _c1|  _c2|            _c3|
+---+-------------------+-----+---------------+
|  1|2013-07-25 00:00:00|11599|         CLOSED|
|  2|2013-07-25 00:00:00|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:00|12111|       COMPLETE|
|  4|2013-07-25 00:00:00| 8827|         CLOSED|
|  5|2013-07-25 00:00:00|11318|       COMPLETE|
+---+-------------------+-----+---------------+
only showing top 5 rows



Thường file csv raw có rất nhiều sạn, chưa clear thì spark có thể định nghĩa sai

Không dùng spark định nghĩa thì tự mình định nghĩa

CÁCH 1:

In [21]:
orders_schema = "order_id long, order_date date, cust_id long, status string"

CÁCH 2:

In [30]:
from pyspark.sql.types import *

In [33]:
orders_schema_struct = StructType([
    StructField("orderid", LongType()),
    StructField("orderdate", DateType()),
    StructField("custid", LongType()),
    StructField("status", StringType())
])

In [28]:
df_my1 = spark.read \
.format("csv") \
.option("header", "false") \
.schema(orders_schema) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")

In [34]:
df_my2 = spark.read \
.format("csv") \
.option("header", "false") \
.schema(orders_schema_struct) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")

In [35]:
df_my1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: string (nullable = true)



In [36]:
df_my2.printSchema()

root
 |-- orderid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- custid: long (nullable = true)
 |-- status: string (nullable = true)



=> Cách viết định nghĩa: cách 2 sẽ phức tạp hơn, nhưng nó chạy nhanh hơn 1 chút >> Tối ưu hơn 1 chút

In [37]:
#Nếu định nghĩa sai trường cuối, string -> long
orders_schema_3 = "order_id long, order_date date, cust_id long, status long"

In [38]:
df_my3 = spark.read \
.format("csv") \
.option("header", "false") \
.schema(orders_schema_3) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")

In [39]:
df_my3.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: long (nullable = true)



In [ ]:
df_my3.show(5)
#Nếu là số thì để long/string thì hiển thị được; nếu là chữ mà mình để long > không match, NULL

+--------+----------+-------+------+
|order_id|order_date|cust_id|status|
+--------+----------+-------+------+
|       1|2013-07-25|  11599|  NULL|
|       2|2013-07-25|    256|  NULL|
|       3|2013-07-25|  12111|  NULL|
|       4|2013-07-25|   8827|  NULL|
|       5|2013-07-25|  11318|  NULL|
+--------+----------+-------+------+
only showing top 5 rows



In [42]:
orders_schema_struct_4 = StructType([
    StructField("orderid", LongType()),
    StructField("orderdate", DateType()),
    StructField("custid", LongType()),
    StructField("status", LongType())
])

In [44]:
df_my4 = spark.read \
.format("csv") \
.option("header", "false") \
.schema(orders_schema_struct_4) \
.load("D:\Learn-spark\learn-spark-maide\orders_2.csv")

In [46]:
df_my4.printSchema()

root
 |-- orderid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- custid: long (nullable = true)
 |-- status: long (nullable = true)



In [48]:
df_my4.show(5)
#Nếu là số thì để long/string thì hiển thị được; nếu là chữ mà mình để long > không match, NULL

+-------+----------+------+------+
|orderid| orderdate|custid|status|
+-------+----------+------+------+
|      1|2013-07-25| 11599|  NULL|
|      2|2013-07-25|   256|  NULL|
|      3|2013-07-25| 12111|  NULL|
|      4|2013-07-25|  8827|  NULL|
|      5|2013-07-25| 11318|  NULL|
+-------+----------+------+------+
only showing top 5 rows

